In [4]:
from pathlib import Path
import json
import numpy as np

# ===============================
# 1. Set project root path
# ===============================
PROJECT_ROOT = Path.cwd().resolve().parent

# Train directory
TRAIN_GEN_DIR = PROJECT_ROOT / "data" / "generated" / "train"

# Feature manifest and normalization stats
TRAIN_FEATURE_MANIFEST = TRAIN_GEN_DIR / "features" / "train_noisy_features_manifest.jsonl"
TRAIN_STATS = TRAIN_GEN_DIR / "features" / "noisy_norm_stats.npz"

print("TRAIN_GEN_DIR:", TRAIN_GEN_DIR)
print("Feature manifest exists:", TRAIN_FEATURE_MANIFEST.exists())
print("Normalization stats exists:", TRAIN_STATS.exists())

# ===============================
# 2. Load feature manifest
# ===============================
rows = []
with open(TRAIN_FEATURE_MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print("\nNumber of training examples:", len(rows))

# ===============================
# 3. Inspect first row
# ===============================
first = rows[0]

print("\nKeys in first row:")
print(first.keys())

print("\nFirst row summary:")
for k, v in first.items():
    if isinstance(v, (str, int, float, bool)):
        print(f"{k}: {v}")

# ===============================
# 4. Check if stacked features exist
# ===============================
has_stacked = "stacked_features_path" in first
print("\nHas stacked features:", has_stacked)

if has_stacked:
    print("Stacked feature dimension (should be 1331):", first.get("stacked_dim"))

# ===============================
# 5. Check a few samples
# ===============================
num_check = min(5, len(rows))

for i in range(num_check):
    row = rows[i]

    print("\n" + "="*60)
    print(f"Sample {i}: ex_id = {row['ex_id']}")

    # Paths
    frame_path = TRAIN_GEN_DIR / row["frame_features_path"]
    label_path = TRAIN_GEN_DIR / row["labels_path"]

    print("Frame feature path exists:", frame_path.exists())
    print("Label path exists:", label_path.exists())

    # Load data
    X_frame = np.load(frame_path)
    y = np.load(label_path)

    # Basic checks
    print("Frame feature shape:", X_frame.shape)
    print("Label shape:", y.shape)
    print("Aligned (X and y same length):", X_frame.shape[0] == y.shape[0])

    print("Frame feature dimension (should be 121):", X_frame.shape[1])
    print("Label unique values:", np.unique(y))

    # Data type
    print("Feature dtype:", X_frame.dtype)
    print("Label dtype:", y.dtype)

    # Numerical stability
    print("NaN count:", np.isnan(X_frame).sum())
    print("Inf count:", np.isinf(X_frame).sum())

    # Statistics
    print("Feature mean:", float(X_frame.mean()))
    print("Feature std:", float(X_frame.std()))

    # Speech ratio
    print("Speech ratio (mean of labels):", float(y.mean()))

    # ===============================
    # Check stacked features
    # ===============================
    if "stacked_features_path" in row:
        stacked_path = TRAIN_GEN_DIR / row["stacked_features_path"]
        stacked_label_path = TRAIN_GEN_DIR / row["stacked_labels_path"]

        print("\nStacked feature path exists:", stacked_path.exists())
        print("Stacked label path exists:", stacked_label_path.exists())

        X_stacked = np.load(stacked_path)
        y_stacked = np.load(stacked_label_path)

        print("Stacked feature shape:", X_stacked.shape)
        print("Stacked label shape:", y_stacked.shape)
        print("Stacked aligned:", X_stacked.shape[0] == y_stacked.shape[0])

        print("Stacked feature dimension (should be 1331):", X_stacked.shape[1])

        print("Stacked NaN count:", np.isnan(X_stacked).sum())
        print("Stacked Inf count:", np.isinf(X_stacked).sum())

        print("Stacked mean:", float(X_stacked.mean()))
        print("Stacked std:", float(X_stacked.std()))

TRAIN_GEN_DIR: /Users/hongjingren/Documents/6140/Noise-Robust-Voice-Activity-Detection-FINAL/data/generated/train
Feature manifest exists: True
Normalization stats exists: True

Number of training examples: 3000

Keys in first row:
dict_keys(['ex_id', 'split', 'sr', 'manifest_type', 'audio_path', 'labels_path', 'frame_features_path', 'num_frames', 'frame_dim', 'frame_params', 'frame_features_normalized', 'stacked_features_path', 'stacked_labels_path', 'stacked_dim', 'context_left', 'context_right'])

First row summary:
ex_id: train_0000000
split: train
sr: 16000
manifest_type: noisy
audio_path: noisy_audio/train_0000000.npy
labels_path: features/noisy_frame_121/train_0000000_y.npy
frame_features_path: features/noisy_frame_121/train_0000000_X.npy
num_frames: 1352
frame_dim: 121
frame_features_normalized: True
stacked_features_path: features/noisy_stacked_1331/train_0000000_X.npy
stacked_labels_path: features/noisy_stacked_1331/train_0000000_y.npy
stacked_dim: 1331
context_left: 5
contex

In [5]:
from pathlib import Path
import json
import numpy as np

# ===============================
# 1. Set project root
# ===============================
PROJECT_ROOT = Path.cwd().resolve().parent
TRAIN_GEN_DIR = PROJECT_ROOT / "data" / "generated" / "train"
TRAIN_FEATURE_MANIFEST = TRAIN_GEN_DIR / "features" / "train_noisy_features_manifest.jsonl"

# ===============================
# 2. Load manifest
# ===============================
rows = []
with open(TRAIN_FEATURE_MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print("Total train examples:", len(rows))

# ===============================
# 3. Initialize counters
# ===============================
num_examples = 0
total_frames = 0
total_stacked_frames = 0

bad_frame_dim = 0
bad_stacked_dim = 0
bad_alignment = 0
bad_stacked_alignment = 0
bad_labels = 0
nan_count_frame = 0
inf_count_frame = 0
nan_count_stacked = 0
inf_count_stacked = 0

sum_frame = 0.0
sumsq_frame = 0.0
count_frame_values = 0

sum_stacked = 0.0
sumsq_stacked = 0.0
count_stacked_values = 0

speech_frames = 0
nonspeech_frames = 0

# ===============================
# 4. Iterate through all examples
# ===============================
for row in rows:
    num_examples += 1

    frame_path = TRAIN_GEN_DIR / row["frame_features_path"]
    label_path = TRAIN_GEN_DIR / row["labels_path"]
    stacked_path = TRAIN_GEN_DIR / row["stacked_features_path"]
    stacked_label_path = TRAIN_GEN_DIR / row["stacked_labels_path"]

    X_frame = np.load(frame_path)
    y = np.load(label_path)
    X_stacked = np.load(stacked_path)
    y_stacked = np.load(stacked_label_path)

    # ---- shape checks ----
    if X_frame.ndim != 2 or X_frame.shape[1] != 121:
        bad_frame_dim += 1

    if X_stacked.ndim != 2 or X_stacked.shape[1] != 1331:
        bad_stacked_dim += 1

    if X_frame.shape[0] != y.shape[0]:
        bad_alignment += 1

    if X_stacked.shape[0] != y_stacked.shape[0]:
        bad_stacked_alignment += 1

    # ---- label check ----
    unique_labels = np.unique(y)
    if not np.all(np.isin(unique_labels, [0, 1])):
        bad_labels += 1

    # ---- NaN / Inf checks ----
    nan_count_frame += np.isnan(X_frame).sum()
    inf_count_frame += np.isinf(X_frame).sum()
    nan_count_stacked += np.isnan(X_stacked).sum()
    inf_count_stacked += np.isinf(X_stacked).sum()

    # ---- global statistics for frame features ----
    sum_frame += X_frame.sum()
    sumsq_frame += np.square(X_frame).sum()
    count_frame_values += X_frame.size

    # ---- global statistics for stacked features ----
    sum_stacked += X_stacked.sum()
    sumsq_stacked += np.square(X_stacked).sum()
    count_stacked_values += X_stacked.size

    # ---- frame counts ----
    total_frames += X_frame.shape[0]
    total_stacked_frames += X_stacked.shape[0]

    # ---- speech ratio ----
    speech_frames += y.sum()
    nonspeech_frames += len(y) - y.sum()

# ===============================
# 5. Compute global mean/std
# ===============================
global_frame_mean = sum_frame / count_frame_values
global_frame_var = (sumsq_frame / count_frame_values) - (global_frame_mean ** 2)
global_frame_std = np.sqrt(max(global_frame_var, 1e-12))

global_stacked_mean = sum_stacked / count_stacked_values
global_stacked_var = (sumsq_stacked / count_stacked_values) - (global_stacked_mean ** 2)
global_stacked_std = np.sqrt(max(global_stacked_var, 1e-12))

speech_ratio = speech_frames / (speech_frames + nonspeech_frames)

# ===============================
# 6. Print summary
# ===============================
print("\n===== FULL TRAIN SANITY CHECK SUMMARY =====")
print("Total examples:", num_examples)
print("Total frame-level frames:", total_frames)
print("Total stacked frames:", total_stacked_frames)

print("\nShape / alignment checks")
print("Bad frame dimension count:", bad_frame_dim)
print("Bad stacked dimension count:", bad_stacked_dim)
print("Bad frame-label alignment count:", bad_alignment)
print("Bad stacked-label alignment count:", bad_stacked_alignment)

print("\nLabel checks")
print("Bad label count:", bad_labels)

print("\nNaN / Inf checks")
print("Frame NaN count:", nan_count_frame)
print("Frame Inf count:", inf_count_frame)
print("Stacked NaN count:", nan_count_stacked)
print("Stacked Inf count:", inf_count_stacked)

print("\nGlobal statistics")
print("Global frame mean:", float(global_frame_mean))
print("Global frame std:", float(global_frame_std))
print("Global stacked mean:", float(global_stacked_mean))
print("Global stacked std:", float(global_stacked_std))

print("\nLabel distribution")
print("Speech frames:", int(speech_frames))
print("Non-speech frames:", int(nonspeech_frames))
print("Speech ratio:", float(speech_ratio))

Total train examples: 3000

===== FULL TRAIN SANITY CHECK SUMMARY =====
Total examples: 3000
Total frame-level frames: 6726129
Total stacked frames: 6726129

Shape / alignment checks
Bad frame dimension count: 0
Bad stacked dimension count: 0
Bad frame-label alignment count: 0
Bad stacked-label alignment count: 0

Label checks
Bad label count: 0

NaN / Inf checks
Frame NaN count: 0
Frame Inf count: 0
Stacked NaN count: 0
Stacked Inf count: 0

Global statistics
Global frame mean: 6.066906088619817e-09
Global frame std: 0.9999997019767761
Global stacked mean: -2.5677611574792536e-06
Global stacked std: 0.9999272227287292

Label distribution
Speech frames: 4733108
Non-speech frames: 1993021
Speech ratio: 0.7036897448740576


In [6]:
from pathlib import Path
import json
import numpy as np

# ===============================
# 1. Set project root
# ===============================
PROJECT_ROOT = Path.cwd().resolve().parent

def check_split(split_name: str):
    split_dir = PROJECT_ROOT / "data" / "generated" / split_name
    manifest_path = split_dir / "features" / f"{split_name}_noisy_features_manifest.jsonl"

    print(f"\n{'='*70}")
    print(f"Checking split: {split_name}")
    print(f"{'='*70}")
    print("Manifest exists:", manifest_path.exists())

    rows = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    print("Total examples:", len(rows))

    bad_frame_dim = 0
    bad_stacked_dim = 0
    bad_alignment = 0
    bad_stacked_alignment = 0
    bad_labels = 0

    nan_count_frame = 0
    inf_count_frame = 0
    nan_count_stacked = 0
    inf_count_stacked = 0

    sum_frame = 0.0
    sumsq_frame = 0.0
    count_frame_values = 0

    sum_stacked = 0.0
    sumsq_stacked = 0.0
    count_stacked_values = 0

    speech_frames = 0
    nonspeech_frames = 0

    for row in rows:
        frame_path = split_dir / row["frame_features_path"]
        label_path = split_dir / row["labels_path"]
        stacked_path = split_dir / row["stacked_features_path"]
        stacked_label_path = split_dir / row["stacked_labels_path"]

        X_frame = np.load(frame_path)
        y = np.load(label_path)
        X_stacked = np.load(stacked_path)
        y_stacked = np.load(stacked_label_path)

        # shape checks
        if X_frame.ndim != 2 or X_frame.shape[1] != 121:
            bad_frame_dim += 1
        if X_stacked.ndim != 2 or X_stacked.shape[1] != 1331:
            bad_stacked_dim += 1
        if X_frame.shape[0] != y.shape[0]:
            bad_alignment += 1
        if X_stacked.shape[0] != y_stacked.shape[0]:
            bad_stacked_alignment += 1

        # label check
        unique_labels = np.unique(y)
        if not np.all(np.isin(unique_labels, [0, 1])):
            bad_labels += 1

        # NaN / Inf
        nan_count_frame += np.isnan(X_frame).sum()
        inf_count_frame += np.isinf(X_frame).sum()
        nan_count_stacked += np.isnan(X_stacked).sum()
        inf_count_stacked += np.isinf(X_stacked).sum()

        # stats
        sum_frame += X_frame.sum()
        sumsq_frame += np.square(X_frame).sum()
        count_frame_values += X_frame.size

        sum_stacked += X_stacked.sum()
        sumsq_stacked += np.square(X_stacked).sum()
        count_stacked_values += X_stacked.size

        speech_frames += y.sum()
        nonspeech_frames += len(y) - y.sum()

    global_frame_mean = sum_frame / count_frame_values
    global_frame_var = (sumsq_frame / count_frame_values) - (global_frame_mean ** 2)
    global_frame_std = np.sqrt(max(global_frame_var, 1e-12))

    global_stacked_mean = sum_stacked / count_stacked_values
    global_stacked_var = (sumsq_stacked / count_stacked_values) - (global_stacked_mean ** 2)
    global_stacked_std = np.sqrt(max(global_stacked_var, 1e-12))

    speech_ratio = speech_frames / (speech_frames + nonspeech_frames)

    print("\nShape / alignment checks")
    print("Bad frame dimension count:", bad_frame_dim)
    print("Bad stacked dimension count:", bad_stacked_dim)
    print("Bad frame-label alignment count:", bad_alignment)
    print("Bad stacked-label alignment count:", bad_stacked_alignment)

    print("\nLabel checks")
    print("Bad label count:", bad_labels)

    print("\nNaN / Inf checks")
    print("Frame NaN count:", nan_count_frame)
    print("Frame Inf count:", inf_count_frame)
    print("Stacked NaN count:", nan_count_stacked)
    print("Stacked Inf count:", inf_count_stacked)

    print("\nGlobal statistics")
    print("Global frame mean:", float(global_frame_mean))
    print("Global frame std:", float(global_frame_std))
    print("Global stacked mean:", float(global_stacked_mean))
    print("Global stacked std:", float(global_stacked_std))

    print("\nLabel distribution")
    print("Speech frames:", int(speech_frames))
    print("Non-speech frames:", int(nonspeech_frames))
    print("Speech ratio:", float(speech_ratio))


check_split("dev")
check_split("test")


Checking split: dev
Manifest exists: True
Total examples: 500

Shape / alignment checks
Bad frame dimension count: 0
Bad stacked dimension count: 0
Bad frame-label alignment count: 0
Bad stacked-label alignment count: 0

Label checks
Bad label count: 0

NaN / Inf checks
Frame NaN count: 0
Frame Inf count: 0
Stacked NaN count: 0
Stacked Inf count: 0

Global statistics
Global frame mean: -0.0006404559826478362
Global frame std: 1.0056949853897095
Global stacked mean: -0.0006451139342971146
Global stacked std: 1.005625605583191

Label distribution
Speech frames: 753074
Non-speech frames: 335379
Speech ratio: 0.6918755334405804

Checking split: test
Manifest exists: True
Total examples: 500

Shape / alignment checks
Bad frame dimension count: 0
Bad stacked dimension count: 0
Bad frame-label alignment count: 0
Bad stacked-label alignment count: 0

Label checks
Bad label count: 0

NaN / Inf checks
Frame NaN count: 0
Frame Inf count: 0
Stacked NaN count: 0
Stacked Inf count: 0

Global statis